In [1]:
"""
================================================================================
IMPROVED β-VAE GDRANK - BALANCED ADVERSARIAL WITH CAREFUL ANNEALING
Key improvements:
1. Gentler adversarial training (doesn't destroy ranking signal)
2. Better β annealing schedule
3. Stronger ranking supervision
4. Adaptive discriminator training
================================================================================
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
import json
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Using device: {device}")

# ============================================================================
# DATA LOADING
# ============================================================================

class YahooLETORDataset:
    def __init__(self, filepath, scaler=None, max_feat_idx=None):
        self.rows = []
        max_idx = 0
        
        with open(filepath, 'r') as f:
            for line in f:
                parsed = self._parse_line(line)
                if parsed:
                    self.rows.append(parsed)
                    qid, rel, feats = parsed
                    if feats:
                        max_idx = max(max_idx, max(feats.keys()))
        
        self.feature_dim = max_idx + 1 if max_feat_idx is None else max_feat_idx
        self._convert_to_matrix()
        
        if scaler is None:
            self.scaler = StandardScaler()
            self.scaler.fit(self.features)
        else:
            self.scaler = scaler
        
        self.features = self.scaler.transform(self.features).astype(np.float32)
        self._group_by_query()
    
    def _parse_line(self, line):
        parts = line.strip().split('#')[0].strip().split()
        if len(parts) < 2:
            return None
        rel = int(parts[0])
        qid = parts[1].split(':')[1]
        feats = {}
        for tok in parts[2:]:
            if ':' in tok:
                idx, val = tok.split(':')
                feats[int(idx) - 1] = float(val)
        return qid, rel, feats
    
    def _convert_to_matrix(self):
        feat_matrix, qids, rels = [], [], []
        for qid, rel, fd in self.rows:
            arr = np.zeros(self.feature_dim, dtype=np.float32)
            for i, v in fd.items():
                if i < self.feature_dim:
                    arr[i] = v
            feat_matrix.append(arr)
            qids.append(qid)
            rels.append(rel)
        self.features = np.stack(feat_matrix, axis=0)
        self.qids = np.array(qids)
        self.rels = np.array(rels)
    
    def _group_by_query(self):
        self.queries = {}
        for i, qid in enumerate(self.qids):
            self.queries.setdefault(qid, []).append(i)

# ============================================================================
# MODELS
# ============================================================================

class Encoder(nn.Module):
    def __init__(self, in_dim, hidden, zdim):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers.extend([nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.2)])
            d = h
        self.net = nn.Sequential(*layers)
        self.mu = nn.Linear(d, zdim)
        self.logvar = nn.Linear(d, zdim)
    
    def forward(self, x):
        h = self.net(x)
        mu = self.mu(h)
        logvar = torch.clamp(self.logvar(h), -10, 10)
        z = mu + (0.5 * logvar).exp() * torch.randn_like(logvar)
        return z, mu, logvar

class Decoder(nn.Module):
    def __init__(self, zr_dim, zb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(zr_dim + zb_dim + 8, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), 
            nn.Linear(64, 1)
        )
    def forward(self, zr, zb, pos_emb):
        return torch.sigmoid(self.net(torch.cat([zr, zb, pos_emb], 1))).squeeze(-1)

class RankingHead(nn.Module):
    def __init__(self, zr_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(zr_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, zr):
        return self.net(zr).squeeze(-1)

# ============================================================================
# MULTI-SCALE DISCRIMINATORS (GENTLER)
# ============================================================================

class MultiScaleDiscriminator(nn.Module):
    """Three discriminators - but we'll use them carefully"""
    def __init__(self, zr_dim, topk=10):
        super().__init__()
        
        # Binary discriminator only (easiest, least destructive)
        self.disc_binary = nn.Sequential(
            nn.Linear(zr_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2)  # Top-5 vs Bottom-5
        )
    
    def forward(self, zr, pos_ids):
        """Returns only binary loss"""
        binary_labels = (pos_ids >= 5).long()
        binary_loss = F.cross_entropy(self.disc_binary(zr), binary_labels)
        return binary_loss

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad):
        return -ctx.lambd * grad, None

def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)

# ============================================================================
# METRICS
# ============================================================================

def ndcg_at_k(true_rels, pred_scores, k=10):
    k = min(k, len(true_rels))
    order = np.argsort(-pred_scores)[:k]
    dcg = np.sum((2**true_rels[order] - 1) / np.log2(np.arange(2, k+2)))
    ideal = np.argsort(-true_rels)[:k]
    idcg = np.sum((2**true_rels[ideal] - 1) / np.log2(np.arange(2, k+2)))
    return dcg / idcg if idcg > 0 else 0.0

def compute_all_ranking_metrics(true_rels, pred_scores, k_values=[1, 3, 5, 10]):
    return {f'NDCG@{k}': ndcg_at_k(true_rels, pred_scores, k) 
            for k in k_values if len(true_rels) >= k}

def aggregate_metrics(metric_list):
    if not metric_list:
        return {}
    agg = defaultdict(list)
    for m in metric_list:
        for k, v in m.items():
            agg[k].append(v)
    return {k: np.mean(v) for k, v in agg.items()}

# ============================================================================
# UTILITIES
# ============================================================================

def free_bits_kl(mu, logvar, free_bits=1.0):
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    return torch.mean(torch.sum(torch.clamp(kl, min=free_bits), 1))

def cov_penalty(zr, zb):
    zr_n = zr - zr.mean(0)
    zb_n = zb - zb.mean(0)
    return torch.sum(((zr_n.T @ zb_n) / (zr.size(0) + 1e-9)) ** 2)

def default_examine_func(k, alpha=0.9):
    return 1.0 / (k ** alpha)

def pbm_clicks_for_impression_list(rels, E_func):
    clicks = []
    for pos, rel in enumerate(rels, 1):
        exam = E_func(pos)
        attr = 1 / (1 + np.exp(-(rel - 1)))
        clicks.append(1 if random.random() < exam * attr else 0)
    return clicks

# ============================================================================
# PAIRWISE RANKING LOSS (STRONGER SUPERVISION)
# ============================================================================

def pairwise_ranking_loss(pred_scores, true_rels, margin=0.5):
    """RankNet-style pairwise loss for better ranking supervision"""
    n = len(pred_scores)
    if n < 2:
        return torch.tensor(0.0, device=pred_scores.device)
    
    loss = 0.0
    count = 0
    for i in range(n):
        for j in range(i+1, n):
            if true_rels[i] != true_rels[j]:
                # If i should be ranked higher than j
                if true_rels[i] > true_rels[j]:
                    diff = pred_scores[j] - pred_scores[i] + margin
                else:
                    diff = pred_scores[i] - pred_scores[j] + margin
                loss += torch.clamp(diff, min=0.0)
                count += 1
    
    return loss / max(count, 1)

# ============================================================================
# IMPROVED TRAINING: BALANCED APPROACH
# ============================================================================

def train_improved_beta_vae(train_file, vali_file, epochs=80, topk=10,
                            max_beta_r=0.6, max_beta_b=0.3, lr=3e-4):
    print("="*100)
    print(" "*15 + "IMPROVED β-VAE: BALANCED ADVERSARIAL + STRONG RANKING")
    print("="*100)
    
    train_set = YahooLETORDataset(train_file)
    vali_set = YahooLETORDataset(vali_file, train_set.scaler, train_set.feature_dim)
    
    print(f"\n📊 {len(train_set.queries)} train, {len(vali_set.queries)} val queries")
    print(f"\n🎯 Key Improvements:")
    print(f"   ✨ z_r dimension: 24 (balanced capacity)")
    print(f"   ✨ Binary discriminator ONLY (gentle debiasing)")
    print(f"   ✨ Stronger ranking head (deeper network)")
    print(f"   ✨ Pairwise ranking loss (better supervision)")
    print(f"   ✨ Careful β annealing: 0→{max_beta_r} over epochs 20→60")
    print(f"   ✨ Adaptive adversarial weight: starts low, grows slowly")
    print(f"   ✨ Discriminator updates: 3x (not 7x - too aggressive)")
    print(f"   ✨ Epochs: {epochs} with patience 30\n")
    
    # Better capacity: 24 dims
    enc_r = Encoder(train_set.feature_dim, [128, 96, 64], 24).to(device)
    enc_b = Encoder(train_set.feature_dim + 8, [64, 48], 16).to(device)
    dec = Decoder(24, 16).to(device)
    disc = MultiScaleDiscriminator(24, topk).to(device)
    pos_emb = nn.Embedding(topk, 8).to(device)
    ranking_head = RankingHead(24).to(device)
    
    opt_g = torch.optim.Adam(
        list(enc_r.parameters()) + list(enc_b.parameters()) + 
        list(dec.parameters()) + list(pos_emb.parameters()) +
        list(ranking_head.parameters()), lr=lr, weight_decay=1e-5
    )
    opt_d = torch.optim.Adam(disc.parameters(), lr=lr * 2, weight_decay=1e-5)
    
    best_ndcg, patience = 0.0, 0
    history = {'train': defaultdict(list), 'val': defaultdict(list)}
    
    for ep in range(epochs):
        # GENTLER β annealing: 20→60
        if ep < 20:
            beta_r, beta_b = 0.0, 0.0
        elif ep >= 60:
            beta_r, beta_b = max_beta_r, max_beta_b
        else:
            prog = (ep - 20) / 40
            beta_r, beta_b = max_beta_r * prog, max_beta_b * prog
        
        # SLOWER adversarial growth
        adv_weight = min(1.0, 0.1 + (ep / epochs) * 0.9)
        
        # Training
        enc_r.train(); enc_b.train(); dec.train(); disc.train(); ranking_head.train()
        losses = defaultdict(list)
        
        for qid, idxs in train_set.queries.items():
            if len(idxs) < 2:
                continue
            
            qf, qr = train_set.features[idxs], train_set.rels[idxs]
            k = min(topk, len(qr))
            order = np.argsort(-(qr + np.random.normal(0, 0.2, len(qr))))[:k]
            clicks = pbm_clicks_for_impression_list(qr[order], default_examine_func)
            
            feats = torch.tensor(qf[order], device=device, dtype=torch.float32)
            clicks_t = torch.tensor(clicks, device=device, dtype=torch.float32)
            rels_t = torch.tensor(qr[order], device=device, dtype=torch.float32)
            pos_ids = torch.clamp(torch.arange(k, device=device), 0, topk-1)
            pos_vecs = pos_emb(pos_ids)
            
            zr, mu_r, logvar_r = enc_r(feats)
            zb, mu_b, logvar_b = enc_b(torch.cat([feats, pos_vecs], 1))
            pred_clicks = dec(zr, zb, pos_vecs)
            pred_rels = ranking_head(zr)
            
            rec_loss = F.binary_cross_entropy(pred_clicks, clicks_t)
            
            # STRONGER ranking supervision (pointwise + pairwise)
            pointwise_loss = F.mse_loss(pred_rels, rels_t)
            pairwise_loss = pairwise_ranking_loss(pred_rels, rels_t)
            rank_loss = pointwise_loss + 0.5 * pairwise_loss
            
            kl_r = free_bits_kl(mu_r, logvar_r, 0.5)  # Lower free bits
            kl_b = free_bits_kl(mu_b, logvar_b, 0.5)
            cov = cov_penalty(zr, zb)
            
            # Train discriminator (only 3x, not 7x)
            for _ in range(3):
                opt_d.zero_grad()
                d_loss = disc(zr.detach(), pos_ids)
                d_loss.backward()
                torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
                opt_d.step()
            
            # Generator with gentle adversarial
            opt_g.zero_grad()
            adv_loss = disc(grad_reverse(zr, adv_weight), pos_ids)
            
            # BALANCED total loss
            total = (rec_loss + 
                    2.0 * rank_loss +  # Emphasize ranking!
                    beta_r * kl_r + beta_b * kl_b + 
                    0.001 * cov + 
                    0.5 * adv_weight * adv_loss)  # Gentle adversarial
            
            total.backward()
            torch.nn.utils.clip_grad_norm_(
                list(enc_r.parameters()) + list(enc_b.parameters()) + 
                list(dec.parameters()) + list(ranking_head.parameters()), 1.0
            )
            opt_g.step()
            
            for k, v in [('total', total), ('rec', rec_loss), ('rank', rank_loss), 
                         ('adv', adv_loss), ('kl_r', kl_r), ('kl_b', kl_b)]:
                losses[k].append(v.item())
        
        for k, v in losses.items():
            history['train'][k].append(np.mean(v))
        
        # Validation
        enc_r.eval(); ranking_head.eval()
        val_metrics = []
        
        with torch.no_grad():
            for qid, idxs in vali_set.queries.items():
                if len(idxs) < 2:
                    continue
                zr, _, _ = enc_r(torch.tensor(vali_set.features[idxs], device=device))
                preds = ranking_head(zr).cpu().numpy()
                val_metrics.append(compute_all_ranking_metrics(vali_set.rels[idxs], preds))
        
        vm = aggregate_metrics(val_metrics)
        for k, v in vm.items():
            history['val'][k].append(v)
        
        ndcg = vm.get('NDCG@10', 0)
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            best_weights = {
                'enc_r': enc_r.state_dict(), 'ranking_head': ranking_head.state_dict(),
                'dec': dec.state_dict(), 'enc_b': enc_b.state_dict()
            }
            best_epoch = ep + 1
            patience = 0
        else:
            patience += 1
        
        if ep % 5 == 0 or ep == epochs - 1:
            print(f"Epoch {ep+1:2d}/{epochs} | β={beta_r:.2f},{beta_b:.2f} adv_w={adv_weight:.2f} | "
                  f"Loss={history['train']['total'][-1]:.3f} | "
                  f"Rank={history['train']['rank'][-1]:.3f} "
                  f"ADV={history['train']['adv'][-1]:.3f} | "
                  f"NDCG@10={ndcg:.4f} {'🏆' if patience == 0 else f'({patience})'}")
        
        if patience >= 30:
            print(f"\n⚠️ Early stopping at epoch {ep+1}")
            break
    
    if best_ndcg > 0:
        enc_r.load_state_dict(best_weights['enc_r'])
        ranking_head.load_state_dict(best_weights['ranking_head'])
        dec.load_state_dict(best_weights['dec'])
        enc_b.load_state_dict(best_weights['enc_b'])
    
    print(f"\n✅ Best NDCG@10: {best_ndcg:.4f} (Epoch {best_epoch})\n")
    
    return enc_r, enc_b, dec, ranking_head, train_set, history

# ============================================================================
# BASELINE
# ============================================================================

def train_baseline(train_file, vali_file, epochs=20):
    print("\n🔬 Training Baseline...")
    train_set = YahooLETORDataset(train_file)
    vali_set = YahooLETORDataset(vali_file, train_set.scaler, train_set.feature_dim)
    
    encoder = nn.Sequential(
        nn.Linear(train_set.feature_dim, 128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 32)
    ).to(device)
    ranking_head = RankingHead(32).to(device)
    opt = torch.optim.Adam(list(encoder.parameters()) + list(ranking_head.parameters()), lr=1e-3)
    
    best_ndcg = 0.0
    history = {'train': defaultdict(list), 'val': defaultdict(list)}
    
    for ep in range(epochs):
        encoder.train(); ranking_head.train()
        losses = []
        
        for qid, idxs in train_set.queries.items():
            if len(idxs) < 2:
                continue
            qf, qr = train_set.features[idxs], train_set.rels[idxs]
            order = np.argsort(-(qr + np.random.normal(0, 0.2, len(qr))))[:10]
            clicks = pbm_clicks_for_impression_list(qr[order], default_examine_func)
            
            feats = torch.tensor(qf[order], device=device, dtype=torch.float32)
            target = torch.tensor(clicks, device=device, dtype=torch.float32)
            
            pred = ranking_head(encoder(feats))
            loss = F.binary_cross_entropy_with_logits(pred, target)
            
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(ranking_head.parameters()), 1.0)
            opt.step()
            losses.append(loss.item())
        
        history['train']['loss'].append(np.mean(losses))
        
        encoder.eval(); ranking_head.eval()
        val_metrics = []
        with torch.no_grad():
            for qid, idxs in vali_set.queries.items():
                if len(idxs) < 2:
                    continue
                feats = torch.tensor(vali_set.features[idxs], device=device)
                preds = ranking_head(encoder(feats)).cpu().numpy()
                val_metrics.append(compute_all_ranking_metrics(vali_set.rels[idxs], preds))
        
        vm = aggregate_metrics(val_metrics)
        for k, v in vm.items():
            history['val'][k].append(v)
        
        ndcg = vm.get('NDCG@10', 0)
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            best_encoder = encoder.state_dict()
            best_rh = ranking_head.state_dict()
        
        if ep % 5 == 0:
            print(f"   Epoch {ep+1}/{epochs} | NDCG@10={ndcg:.4f}")
    
    if best_ndcg > 0:
        encoder.load_state_dict(best_encoder)
        ranking_head.load_state_dict(best_rh)
    
    print(f"✅ Best NDCG@10: {best_ndcg:.4f}\n")
    return encoder, ranking_head, train_set, history





# ============================================================================
# IPS RANKER BASELINE
# ============================================================================

def train_ips_ranker(train_file, vali_file, epochs=30, topk=10):
    """
    IPS (Inverse Propensity Scoring) Ranker
    Trains a ranker by reweighting clicks with inverse propensity scores
    to correct for position bias.
    """
    print("\n🔬 Training IPS Ranker...")
    
    train_set = YahooLETORDataset(train_file)
    vali_set = YahooLETORDataset(vali_file, train_set.scaler, train_set.feature_dim)
    
    # Simple neural ranker architecture
    encoder = nn.Sequential(
        nn.Linear(train_set.feature_dim, 128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 32)
    ).to(device)
    
    ranking_head = RankingHead(32).to(device)
    
    opt = torch.optim.Adam(
        list(encoder.parameters()) + list(ranking_head.parameters()), 
        lr=1e-3, weight_decay=1e-5
    )
    
    best_ndcg = 0.0
    patience_counter = 0
    history = {'train': defaultdict(list), 'val': defaultdict(list)}
    
    for ep in range(epochs):
        encoder.train()
        ranking_head.train()
        losses = []
        
        for qid, idxs in train_set.queries.items():
            if len(idxs) < 2:
                continue
            
            qf, qr = train_set.features[idxs], train_set.rels[idxs]
            k = min(topk, len(qr))
            
            # Generate ranking with noise
            order = np.argsort(-(qr + np.random.normal(0, 0.2, len(qr))))[:k]
            
            # Simulate clicks
            clicks = pbm_clicks_for_impression_list(qr[order], default_examine_func)
            
            feats = torch.tensor(qf[order], device=device, dtype=torch.float32)
            clicks_t = torch.tensor(clicks, device=device, dtype=torch.float32)
            
            # Compute IPS weights
            # Propensity = P(examination|position) = 1 / k^alpha
            positions = torch.arange(1, k+1, device=device, dtype=torch.float32)
            propensities = 1.0 / (positions ** 0.9)  # alpha=0.9
            ips_weights = 1.0 / propensities  # Inverse propensity
            
            # Normalize weights to prevent explosion
            ips_weights = ips_weights / ips_weights.sum() * k
            
            # Forward pass
            pred = ranking_head(encoder(feats)).squeeze(-1)
            
            # IPS-weighted binary cross-entropy
            # Only weight the positive (clicked) samples
            loss = F.binary_cross_entropy_with_logits(
                pred, clicks_t, weight=ips_weights, reduction='mean'
            )
            
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(ranking_head.parameters()), 1.0
            )
            opt.step()
            
            losses.append(loss.item())
        
        history['train']['loss'].append(np.mean(losses))
        
        # Validation
        encoder.eval()
        ranking_head.eval()
        val_metrics = []
        
        with torch.no_grad():
            for qid, idxs in vali_set.queries.items():
                if len(idxs) < 2:
                    continue
                feats = torch.tensor(vali_set.features[idxs], device=device)
                preds = ranking_head(encoder(feats)).cpu().numpy()
                val_metrics.append(
                    compute_all_ranking_metrics(vali_set.rels[idxs], preds)
                )
        
        vm = aggregate_metrics(val_metrics)
        for k, v in vm.items():
            history['val'][k].append(v)
        
        ndcg = vm.get('NDCG@10', 0)
        
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            best_encoder = encoder.state_dict()
            best_rh = ranking_head.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if ep % 5 == 0:
            print(f"   Epoch {ep+1}/{epochs} | Loss={history['train']['loss'][-1]:.3f} | "
                  f"NDCG@10={ndcg:.4f} {'🏆' if patience_counter == 0 else f'({patience_counter})'}")
        
        if patience_counter >= 15:
            print(f"   Early stopping at epoch {ep+1}")
            break
    
    if best_ndcg > 0:
        encoder.load_state_dict(best_encoder)
        ranking_head.load_state_dict(best_rh)
    
    print(f"✅ Best NDCG@10: {best_ndcg:.4f}\n")
    
    return encoder, ranking_head, train_set, history



# ============================================================================
# REGRESSION-EM BASELINE [Wang et al., WSDM 2018]
# ============================================================================

def train_regression_em(train_file, vali_file, epochs=20, em_iterations=10, topk=10):
    """
    Regression-based EM for position bias estimation
    Based on: Wang et al., "Position Bias Estimation for Unbiased Learning to Rank 
    in Personal Search", WSDM 2018
    
    Key idea: Use EM to jointly estimate:
    - Position bias θ_k (examination probability at position k)
    - Relevance γ_{q,d} via regression function f(x) instead of exact IDs
    """
    print("\n🔬 Training Regression-EM...")
    
    train_set = YahooLETORDataset(train_file)
    vali_set = YahooLETORDataset(vali_file, train_set.scaler, train_set.feature_dim)
    
    # Initialize position bias uniformly
    theta = {k: 1.0 / (k + 1) for k in range(topk)}  # Simple initialization
    
    # Relevance predictor (GBDT-based)
    from sklearn.ensemble import GradientBoostingClassifier
    relevance_model = None
    
    print(f"   Running {em_iterations} EM iterations...")
    
    # Collect training data for EM
    em_data = []
    for qid, idxs in train_set.queries.items():
        if len(idxs) < 2:
            continue
        
        qf, qr = train_set.features[idxs], train_set.rels[idxs]
        k = min(topk, len(qr))
        
        # Generate ranking with noise
        order = np.argsort(-(qr + np.random.normal(0, 0.2, len(qr))))[:k]
        
        # Simulate clicks
        clicks = pbm_clicks_for_impression_list(qr[order], default_examine_func)
        
        for pos, (doc_idx, click) in enumerate(zip(order, clicks)):
            em_data.append({
                'features': qf[doc_idx],
                'click': click,
                'position': pos,
                'relevance': qr[doc_idx]  # For validation only
            })
    
    # EM Algorithm
    for em_iter in range(em_iterations):
        # ==================== E-STEP ====================
        # Estimate hidden variables E (examination) and R (relevance)
        
        for data_point in em_data:
            c = data_point['click']
            k = data_point['position']
            
            if c == 1:
                # If clicked, we know E=1 and R=1
                data_point['P_E'] = 1.0
                data_point['P_R'] = 1.0
            else:
                # If not clicked, estimate probabilities
                # P(E=1|C=0) and P(R=1|C=0) based on current parameters
                
                # Get relevance prediction from current model
                if relevance_model is not None:
                    gamma = relevance_model.predict_proba([data_point['features']])[0, 1]
                else:
                    gamma = 0.5  # Initial guess
                
                theta_k = theta.get(k, 0.1)
                
                # From Eq. 1 in the paper:
                # P(E=1|C=0) = θ_k(1-γ) / (1 - θ_k*γ)
                # P(R=1|C=0) = (1-θ_k)γ / (1 - θ_k*γ)
                
                denom = 1 - theta_k * gamma
                if denom > 0.001:  # Avoid division by zero
                    data_point['P_E'] = (theta_k * (1 - gamma)) / denom
                    data_point['P_R'] = ((1 - theta_k) * gamma) / denom
                else:
                    data_point['P_E'] = 0.5
                    data_point['P_R'] = 0.5
        
        # ==================== M-STEP ====================
        # Update parameters
        
        # 1. Update position bias θ_k (Eq. 2 in paper)
        position_counts = defaultdict(lambda: {'examined': 0.0, 'total': 0.0})
        
        for data_point in em_data:
            k = data_point['position']
            c = data_point['click']
            p_e = data_point['P_E']
            
            position_counts[k]['examined'] += c + (1 - c) * p_e
            position_counts[k]['total'] += 1.0
        
        for k in range(topk):
            if position_counts[k]['total'] > 0:
                theta[k] = position_counts[k]['examined'] / position_counts[k]['total']
            else:
                theta[k] = 1.0 / (k + 2)  # Fallback
        
        # 2. Update relevance model f(x) via regression
        # Sample relevance labels based on P(R=1|...)
        X_train = []
        y_train = []
        
        for data_point in em_data:
            X_train.append(data_point['features'])
            
            # Sample binary label from P(R=1)
            if np.random.random() < data_point['P_R']:
                y_train.append(1)
            else:
                y_train.append(0)
        
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        
        # Train GBDT relevance model
        relevance_model = GradientBoostingClassifier(
            n_estimators=50,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
            warm_start=(em_iter > 0)  # Continue from previous
        )
        relevance_model.fit(X_train, y_train)
        
        if (em_iter + 1) % 3 == 0 or em_iter == 0:
            print(f"   EM Iteration {em_iter + 1}/{em_iterations} | "
                  f"θ=[{theta[0]:.3f}, {theta[1]:.3f}, {theta[2]:.3f}...]")
    
    print(f"   EM converged. Final position bias: {[f'{theta[k]:.3f}' for k in range(5)]}")
    
    # Train final ranking model using estimated propensities
    print("   Training final ranker with estimated propensities...")
    
    encoder = nn.Sequential(
        nn.Linear(train_set.feature_dim, 128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 32)
    ).to(device)
    
    ranking_head = RankingHead(32).to(device)
    
    opt = torch.optim.Adam(
        list(encoder.parameters()) + list(ranking_head.parameters()), 
        lr=1e-3, weight_decay=1e-5
    )
    
    best_ndcg = 0.0
    patience_counter = 0
    history = {'train': defaultdict(list), 'val': defaultdict(list)}
    
    for ep in range(epochs):
        encoder.train()
        ranking_head.train()
        losses = []
        
        for qid, idxs in train_set.queries.items():
            if len(idxs) < 2:
                continue
            
            qf, qr = train_set.features[idxs], train_set.rels[idxs]
            k = min(topk, len(qr))
            
            order = np.argsort(-(qr + np.random.normal(0, 0.2, len(qr))))[:k]
            clicks = pbm_clicks_for_impression_list(qr[order], default_examine_func)
            
            feats = torch.tensor(qf[order], device=device, dtype=torch.float32)
            clicks_t = torch.tensor(clicks, device=device, dtype=torch.float32)
            
            # IPS weights from EM-estimated propensities
            ips_weights = torch.tensor(
                [1.0 / theta.get(pos, 0.1) for pos in range(k)],
                device=device, dtype=torch.float32
            )
            
            pred = ranking_head(encoder(feats)).squeeze(-1)
            
            # IPS-weighted loss
            loss = F.binary_cross_entropy_with_logits(
                pred, clicks_t, weight=ips_weights, reduction='mean'
            )
            
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(ranking_head.parameters()), 1.0
            )
            opt.step()
            
            losses.append(loss.item())
        
        history['train']['loss'].append(np.mean(losses))
        
        # Validation
        encoder.eval()
        ranking_head.eval()
        val_metrics = []
        
        with torch.no_grad():
            for qid, idxs in vali_set.queries.items():
                if len(idxs) < 2:
                    continue
                feats = torch.tensor(vali_set.features[idxs], device=device)
                preds = ranking_head(encoder(feats)).cpu().numpy()
                val_metrics.append(
                    compute_all_ranking_metrics(vali_set.rels[idxs], preds)
                )
        
        vm = aggregate_metrics(val_metrics)
        for k, v in vm.items():
            history['val'][k].append(v)
        
        ndcg = vm.get('NDCG@10', 0)
        
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            best_encoder = encoder.state_dict()
            best_rh = ranking_head.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if ep % 5 == 0:
            print(f"   Epoch {ep+1}/{epochs} | Loss={history['train']['loss'][-1]:.3f} | "
                  f"NDCG@10={ndcg:.4f} {'🏆' if patience_counter == 0 else f'({patience_counter})'}")
        
        if patience_counter >= 10:
            print(f"   Early stopping at epoch {ep+1}")
            break
    
    if best_ndcg > 0:
        encoder.load_state_dict(best_encoder)
        ranking_head.load_state_dict(best_rh)
    
    print(f"✅ Best NDCG@10: {best_ndcg:.4f}\n")
    
    return encoder, ranking_head, train_set, history, theta
    

# ============================================================================
# COUNTERFACTUAL EVALUATION
# ============================================================================

def evaluate_counterfactual(enc_vae, rh_vae, enc_base, rh_base, test_set):
    print("\n" + "="*100)
    print(" "*30 + "COUNTERFACTUAL EVALUATION")
    print("="*100)
    
    def ips_eval(enc, rh, name):
        enc.eval(); rh.eval()
        total_ips, total_dcg, n = 0, 0, 0
        
        with torch.no_grad():
            for qid, idxs in test_set.queries.items():
                if len(idxs) < 2:
                    continue
                feats = torch.tensor(test_set.features[idxs], device=device, dtype=torch.float32)
                rels = test_set.rels[idxs]
                
                if hasattr(enc, 'mu'):
                    z, _, _ = enc(feats)
                else:
                    z = enc(feats)
                
                preds = rh(z).cpu().numpy()
                ranking = np.argsort(-preds)
                clicks = pbm_clicks_for_impression_list(rels[ranking], default_examine_func)
                
                ips_dcg = sum((click / default_examine_func(pos)) * (2**rels[doc] - 1) / np.log2(pos + 1)
                             for pos, (doc, click) in enumerate(zip(ranking, clicks), 1) if click > 0)
                dcg = sum((2**rels[doc] - 1) / np.log2(pos + 1) 
                         for pos, doc in enumerate(ranking[:10], 1))
                
                total_ips += ips_dcg
                total_dcg += dcg
                n += 1
        
        return total_ips / n, total_dcg / n, n
    
    print("\n🎯 Inverse Propensity Scoring")
    print("-" * 80)
    ips_vae, dcg_vae, n = ips_eval(enc_vae, rh_vae, "Improved β-VAE")
    print(f"Improved β-VAE: IPS-DCG={ips_vae:.4f}, DCG={dcg_vae:.4f} ({n} queries)")
    
    ips_base, dcg_base, n = ips_eval(enc_base, rh_base, "Baseline")
    print(f"Baseline:       IPS-DCG={ips_base:.4f}, DCG={dcg_base:.4f} ({n} queries)")
    
    imp = ((ips_vae - ips_base) / ips_base * 100) if ips_base > 0 else 0
    print(f"\n{'🎉 POSITIVE' if imp > 0 else '❌ NEGATIVE'} IPS Improvement: {imp:+.2f}%")
    
    if imp > 0:
        print("   ✅ SUCCESS! Balanced approach worked!")
        print("   The model learned debiased relevance while maintaining ranking quality!")
    else:
        print("   The model still needs fine-tuning.")
        print("   But performance should be better than before.")
    
    return {
        'ips_vae': ips_vae, 'ips_base': ips_base, 'improvement': imp,
        'dcg_vae': dcg_vae, 'dcg_base': dcg_base
    }

# ============================================================================
# TEST EVALUATION
# ============================================================================

def evaluate_test(test_file, encoder, ranking_head, train_set):
    test_set = YahooLETORDataset(test_file, train_set.scaler, train_set.feature_dim)
    
    encoder.eval(); ranking_head.eval()
    metrics = []
    
    with torch.no_grad():
        for qid, idxs in test_set.queries.items():
            if len(idxs) < 2:
                continue
            feats = torch.tensor(test_set.features[idxs], device=device)
            
            if hasattr(encoder, 'mu'):
                z, _, _ = encoder(feats)
            else:
                z = encoder(feats)
            
            preds = ranking_head(z).cpu().numpy()
            metrics.append(compute_all_ranking_metrics(test_set.rels[idxs], preds))
    
    results = aggregate_metrics(metrics)
    
    print(f"\n📊 Test Results:")
    for k in ['NDCG@1', 'NDCG@3', 'NDCG@5', 'NDCG@10']:
        if k in results:
            print(f"   {k}: {results[k]:.4f}")
    
    return results, test_set


def compute_ips_dcg(encoder, ranking_head, test_set):
    """
    Compute IPS-DCG for a given encoder and ranking head
    Returns: (ips_dcg, standard_dcg, num_queries)
    """
    encoder.eval()
    ranking_head.eval()
    
    total_ips = 0
    total_dcg = 0
    n = 0
    
    with torch.no_grad():
        for qid, idxs in test_set.queries.items():
            if len(idxs) < 2:
                continue
            
            feats = torch.tensor(test_set.features[idxs], device=device, dtype=torch.float32)
            rels = test_set.rels[idxs]
            
            # Get predictions
            if hasattr(encoder, 'mu'):  # VAE encoder
                z, _, _ = encoder(feats)
            else:  # Standard encoder
                z = encoder(feats)
            
            preds = ranking_head(z).cpu().numpy()
            
            # Rank documents
            ranking = np.argsort(-preds)
            
            # Simulate clicks
            clicks = pbm_clicks_for_impression_list(rels[ranking], default_examine_func)
            
            # Compute IPS-DCG
            ips_dcg = sum(
                (click / default_examine_func(pos)) * (2**rels[doc] - 1) / np.log2(pos + 1)
                for pos, (doc, click) in enumerate(zip(ranking, clicks), 1) 
                if click > 0
            )
            
            # Compute standard DCG
            dcg = sum(
                (2**rels[doc] - 1) / np.log2(pos + 1) 
                for pos, doc in enumerate(ranking[:10], 1)
            )
            
            total_ips += ips_dcg
            total_dcg += dcg
            n += 1
    
    return total_ips / n, total_dcg / n, n    

# ============================================================================
# MAIN EXPERIMENT
# ============================================================================

def run_complete_experiment(train_file, vali_file, test_file):
    print("\n" + "="*100)
    print(" "*25 + "COMPLETE EXPERIMENT WITH ALL BASELINES")
    print("="*100)
    
    # Train all models
    print("\n" + "="*100)
    print(" "*35 + "TRAINING PHASE")
    print("="*100)
    
    # 1. Baseline (No Debiasing)
    print("\n[1/4] Training Baseline (No Debiasing)...")
    enc_base, rh_base, ts_base, hist_base = train_baseline(
        train_file, vali_file, epochs=20
    )
    
    # 2. IPS Ranker
    print("\n[2/4] Training IPS Ranker...")
    enc_ips, rh_ips, ts_ips, hist_ips = train_ips_ranker(
        train_file, vali_file, epochs=30
    )
    
    # 3. Regression-EM
    print("\n[3/4] Training Regression-EM...")
    enc_em, rh_em, ts_em, hist_em, theta_em = train_regression_em(
        train_file, vali_file, epochs=20, em_iterations=10
    )
    
    # 4. Your Improved β-VAE
    print("\n[4/4] Training Improved β-VAE GDRank...")
    enc_vae, enc_b, dec, rh_vae, train_set, hist_vae = train_improved_beta_vae(
        train_file, vali_file, 
        epochs=80,
        max_beta_r=0.6,
        max_beta_b=0.3,
        lr=3e-4
    )
    
    # Test Evaluation
    print("\n" + "="*100)
    print(" "*35 + "TEST EVALUATION")
    print("="*100)
    
    print("\n🔬 Baseline (No Debiasing):")
    results_base, test_set = evaluate_test(test_file, enc_base, rh_base, ts_base)
    
    print("\n🔬 IPS Ranker:")
    results_ips, _ = evaluate_test(test_file, enc_ips, rh_ips, ts_ips)
    
    print("\n🔬 Regression-EM:")
    results_em, _ = evaluate_test(test_file, enc_em, rh_em, ts_em)
    
    print("\n🔬 Improved β-VAE GDRank:")
    results_vae, _ = evaluate_test(test_file, enc_vae, rh_vae, train_set)
    
    # Counterfactual Evaluation
    print("\n" + "="*100)
    print(" "*30 + "COUNTERFACTUAL EVALUATION")
    print("="*100)
    
    print("\n🎯 IPS-DCG Evaluation for All Models:")
    print("-" * 80)
    
    # Baseline
    enc_base.eval(); rh_base.eval()
    ips_base, dcg_base, n = compute_ips_dcg(enc_base, rh_base, test_set)
    print(f"Baseline:           IPS-DCG={ips_base:.4f}, DCG={dcg_base:.4f} ({n} queries)")
    
    # IPS Ranker
    enc_ips.eval(); rh_ips.eval()
    ips_ips_model, dcg_ips_model, n = compute_ips_dcg(enc_ips, rh_ips, test_set)
    print(f"IPS Ranker:         IPS-DCG={ips_ips_model:.4f}, DCG={dcg_ips_model:.4f} ({n} queries)")
    
    # Regression-EM
    enc_em.eval(); rh_em.eval()
    ips_em_model, dcg_em_model, n = compute_ips_dcg(enc_em, rh_em, test_set)
    print(f"Regression-EM:      IPS-DCG={ips_em_model:.4f}, DCG={dcg_em_model:.4f} ({n} queries)")
    
    # β-VAE
    enc_vae.eval(); rh_vae.eval()
    ips_vae, dcg_vae, n = compute_ips_dcg(enc_vae, rh_vae, test_set)
    print(f"β-VAE (Ours):       IPS-DCG={ips_vae:.4f}, DCG={dcg_vae:.4f} ({n} queries)")
    
    # Summary Table
    print("\n" + "="*100)
    print(" "*35 + "FINAL SUMMARY")
    print("="*100)
    
    print(f"\n{'Model':<25} {'NDCG@10':<12} {'IPS-DCG':<12} {'IPS Improvement'}")
    print("-" * 80)
    
    base_ndcg = results_base.get('NDCG@10', 0)
    ips_ndcg = results_ips.get('NDCG@10', 0)
    em_ndcg = results_em.get('NDCG@10', 0)
    vae_ndcg = results_vae.get('NDCG@10', 0)
    
    ips_imp_ips = ((ips_ips_model - ips_base) / ips_base * 100) if ips_base > 0 else 0
    ips_imp_em = ((ips_em_model - ips_base) / ips_base * 100) if ips_base > 0 else 0
    ips_imp_vae = ((ips_vae - ips_base) / ips_base * 100) if ips_base > 0 else 0
    
    print(f"{'Baseline (No Debiasing)':<25} {base_ndcg:<12.4f} {ips_base:<12.4f} {'baseline'}")
    print(f"{'IPS Ranker':<25} {ips_ndcg:<12.4f} {ips_ips_model:<12.4f} {ips_imp_ips:+.2f}%")
    print(f"{'Regression-EM':<25} {em_ndcg:<12.4f} {ips_em_model:<12.4f} {ips_imp_em:+.2f}%")
    print(f"{'β-VAE GDRank (Ours)':<25} {vae_ndcg:<12.4f} {ips_vae:<12.4f} {ips_imp_vae:+.2f}%")
    
    print("\n" + "="*100)
    if ips_imp_vae > max(ips_imp_ips, ips_imp_em):
        print("🎉 SUCCESS! Our β-VAE GDRank outperforms all baselines!")
        print(f"   - Best IPS improvement: {ips_imp_vae:+.2f}%")
    else:
        print("📊 Results summary:")
        print(f"   - β-VAE IPS improvement: {ips_imp_vae:+.2f}%")
        print(f"   - Best baseline: {max(ips_imp_ips, ips_imp_em):+.2f}%")
    print("="*100)
    
    return {
        'baseline': results_base,
        'ips': results_ips,
        'regression_em': results_em,
        'vae': results_vae,
        'ips_scores': {
            'baseline': ips_base,
            'ips_ranker': ips_ips_model,
            'regression_em': ips_em_model,
            'vae': ips_vae
        }
    }
# ============================================================================
# RUN IT!
# ============================================================================

if __name__ == "__main__":
    '''# SET YOUR PATHS HERE
    train_file = "C:\\Users\\amala\\Desktop\\ltrc_yahoo\\set2.train.txt"
    vali_file = "C:\\Users\\amala\\Desktop\\ltrc_yahoo\\set2.valid.txt"
    test_file = "C:\\Users\\amala\\Desktop\\ltrc_yahoo\\set2.test.txt"'''

     # File paths
    train_file = "C:\\Users\\Amala K J\\Desktop\\yahoo\\set2.train.txt"
    vali_file = "C:\\Users\\Amala K J\\Desktop\\yahoo\\set2.valid.txt"
    test_file = "C:\\Users\\Amala K J\\Desktop\\yahoo\\set2.test.txt"
    
    all_results = run_complete_experiment(train_file, vali_file, test_file)
    
    print("\n" + "="*100)
    print("🎉 EXPERIMENT COMPLETE!")
    print("="*100)
    
    # Save results to JSON
    results_summary = {
        'baseline_ndcg10': all_results['baseline'].get('NDCG@10', 0),
        'ips_ndcg10': all_results['ips'].get('NDCG@10', 0),
        'vae_ndcg10': all_results['vae'].get('NDCG@10', 0),
        'baseline_ips_dcg': all_results['ips_scores']['baseline'],
        'ips_ranker_ips_dcg': all_results['ips_scores']['ips_ranker'],
        'vae_ips_dcg': all_results['ips_scores']['vae']
    }
    
    with open('results_summary.json', 'w') as f:
        json.dump(results_summary, f, indent=2)
    
    print("\n📁 Results saved to 'results_summary.json'")

🚀 Using device: cuda

                         COMPLETE EXPERIMENT WITH ALL BASELINES

                                   TRAINING PHASE

[1/4] Training Baseline (No Debiasing)...

🔬 Training Baseline...
   Epoch 1/20 | NDCG@10=0.6695
   Epoch 6/20 | NDCG@10=0.6691
   Epoch 11/20 | NDCG@10=0.6926
   Epoch 16/20 | NDCG@10=0.6803
✅ Best NDCG@10: 0.6926


[2/4] Training IPS Ranker...

🔬 Training IPS Ranker...
   Epoch 1/30 | Loss=0.445 | NDCG@10=0.6453 🏆
   Epoch 6/30 | Loss=0.419 | NDCG@10=0.6563 🏆
   Epoch 11/30 | Loss=0.411 | NDCG@10=0.6706 (1)
   Epoch 16/30 | Loss=0.409 | NDCG@10=0.6794 (3)
   Epoch 21/30 | Loss=0.419 | NDCG@10=0.6479 (8)
   Epoch 26/30 | Loss=0.401 | NDCG@10=0.6722 (13)
✅ Best NDCG@10: 0.6877


[3/4] Training Regression-EM...

🔬 Training Regression-EM...
   Running 10 EM iterations...
   EM Iteration 1/10 | θ=[1.000, 0.602, 0.406...]
   EM Iteration 3/10 | θ=[1.000, 0.652, 0.432...]
   EM Iteration 6/10 | θ=[1.000, 0.643, 0.416...]
   EM Iteration 9/10 | θ=[1.000, 0